In [ ]:
# Unsupervised Model
# This notebook contains:
# - K-Means Clustering

In [2]:
# =============================================================
# IMPORT LIBRARIES
# =============================================================
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, BisectingKMeans, DBSCAN
from sklearn.metrics import silhouette_score, rand_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler


# =============================================================
# LOAD AND PREPARE DATA
# =============================================================
df = pd.read_csv("/kaggle/input/datasets/bharatgarg1234/diabetes-dataset/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
# Features and labels
X = df.drop(columns=["Diabetes_binary"]).values
y = df["Diabetes_binary"].values

# Standardize features (IMPORTANT for clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Number of clusters (from true labels)
k = len(np.unique(y))


# =============================================================
# CLASSICAL K-MEANS (Random Init)
# =============================================================
kmeans_random = KMeans(
    n_clusters=k,
    init="random",
    n_init=10,
    random_state=42
)

labels_random = kmeans_random.fit_predict(X_scaled)

sil_random = silhouette_score(X_scaled, labels_random)
rand_random = rand_score(y, labels_random)
ari_random = adjusted_rand_score(y, labels_random)


# =============================================================
# K-MEANS++
# =============================================================
# kmeans_pp = KMeans(
#     n_clusters=k,
#     init="k-means++",
#     n_init=10,
#     random_state=42
# )

# labels_pp = kmeans_pp.fit_predict(X_scaled)

# sil_pp = silhouette_score(X_scaled, labels_pp)
# rand_pp = rand_score(y, labels_pp)


# =============================================================
# BISECTING K-MEANS
# =============================================================
# bisect_kmeans = BisectingKMeans(
#     n_clusters=k,
#     random_state=42
# )

# labels_bisect = bisect_kmeans.fit_predict(X_scaled)

# sil_bisect = silhouette_score(X_scaled, labels_bisect)
# rand_bisect = rand_score(y, labels_bisect)


# =============================================================
# DBSCAN
# =============================================================
# dbscan = DBSCAN(
#     eps=0.5,        # you can tune this
#     min_samples=5
# )

# labels_db = dbscan.fit_predict(X_scaled)

# # Handle noise points (-1)
# mask = labels_db != -1

# # Silhouette only works if >1 cluster
# if len(set(labels_db[mask])) > 1:
#     sil_db = silhouette_score(X_scaled[mask], labels_db[mask])
# else:
#     sil_db = -1  # invalid case

# rand_db = rand_score(y, labels_db)


# =============================================================
# PRINT RESULTS
# =============================================================
print("Clustering Performance:\n")

print("Classical K-Means (Random Init)")
print("Silhouette:", sil_random)
print("Rand Index:", rand_random)
print("ARI:", ari_random)
print()

# print("K-Means++")
# print("Silhouette:", sil_pp)
# print("Rand Index:", rand_pp)
# print()

# print("Bisecting K-Means")
# print("Silhouette:", sil_bisect)
# print("Rand Index:", rand_bisect)
# print()

# print("DBSCAN")
# print("Silhouette:", sil_db)
# print("Rand Index:", rand_db)
# print()


Clustering Performance:

Classical K-Means (Random Init)
Silhouette: 0.16335016983976272
Rand Index: 0.556477322156718
ARI: 0.11295602753290696



In [3]:
# =============================================================
# K-MEANS FOR MULTIPLE k VALUES (Corrected)
# =============================================================

k_values = [2, 3, 4, 5]
results_k = []

for k in k_values:
    kmeans_random = KMeans(
        n_clusters=k,
        init="random",
        n_init=10,
        random_state=42
    )
    
    labels_k = kmeans_random.fit_predict(X_scaled)   
    
    sil = silhouette_score(X_scaled, labels_k)
    ari = adjusted_rand_score(y, labels_k)          
    
    results_k.append([k, sil, ari])

df_kmeans_results = pd.DataFrame(results_k, columns=["k", "Silhouette", "ARI"])

print("K-Means Results for Different k Values:\n")
print(df_kmeans_results)


K-Means Results for Different k Values:

   k  Silhouette       ARI
0  2    0.163350  0.112956
1  3    0.075409  0.149577
2  4    0.082853  0.136574
3  5    0.090381  0.123380


In [4]:
import pandas as pd
import numpy as np

# Your results
results = pd.DataFrame({
    "Model": ["KMeans_Random", "KMeans_PP", "Bisecting_KMeans", "DBSCAN"],
    "Silhouette": [0.16335, 0.16332, 0.16394, 0.16399],
    "RandIndex": [0.55648, 0.55651, 0.55607, 0.50543]
})

# -----------------------------
# 1. WIN COUNT METHOD
# -----------------------------
win_count = {"Model": [], "Wins": []}

for metric in ["Silhouette", "RandIndex"]:
    best_value = results[metric].max()
    best_model = results.loc[results[metric].idxmax(), "Model"]
    
    # Add win to that model
    win_count["Model"].append(best_model)
    win_count["Wins"].append(metric)

# Convert to table
win_df = pd.DataFrame(win_count)
win_summary = win_df["Model"].value_counts().reset_index()
win_summary.columns = ["Model", "WinCount"]

# -----------------------------
# 2. RELATIVE PERFORMANCE INDEX (RPI)
# -----------------------------
def compute_rpi(column):
    worst = results[column].min()
    return results[column] - worst   # worst gets 0

results["RPI_Silhouette"] = compute_rpi("Silhouette")
results["RPI_Rand"] = compute_rpi("RandIndex")
results["RPI_Total"] = results["RPI_Silhouette"] + results["RPI_Rand"]

# -----------------------------
# FINAL OUTPUT
# -----------------------------
print("=== WIN COUNT ===")
print(win_summary, "\n")

print("=== RPI TABLE ===")
print(results[["Model", "RPI_Silhouette", "RPI_Rand", "RPI_Total"]])


=== WIN COUNT ===
       Model  WinCount
0     DBSCAN         1
1  KMeans_PP         1 

=== RPI TABLE ===
              Model  RPI_Silhouette  RPI_Rand  RPI_Total
0     KMeans_Random         0.00003   0.05105    0.05108
1         KMeans_PP         0.00000   0.05108    0.05108
2  Bisecting_KMeans         0.00062   0.05064    0.05126
3            DBSCAN         0.00067   0.00000    0.00067
